# Stage A: photoreal Gaussian splat of the bee from the orbit videos

**Runtime:** GPU, A100 (Runtime → Change runtime type). ~30-45 min total.

Pipeline: orbit clips → frames (5 fps) → drop blurry frames → COLMAP structure-from-motion (camera poses) →
nerfstudio **splatfacto** (3D Gaussian Splatting, Kerbl et al. 2023, via gsplat) → export `.ply` + an orbit preview video → Drive.

Before running: put the 5 orbit `.mp4` files in `MyDrive/dirac_bee/orbit/` (keep the original names).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
DRIVE = '/content/drive/MyDrive/dirac_bee'
CLIPS = sorted(glob.glob(f'{DRIVE}/orbit/*.mp4'))
print(len(CLIPS), 'clips:', [os.path.basename(c) for c in CLIPS])
assert len(CLIPS) > 0, 'put the orbit videos in MyDrive/dirac_bee/orbit/'

## 1. Install
COLMAP from apt (CPU build; fine for ~300 images) and nerfstudio (pulls gsplat; its CUDA kernels compile on first use, ~5 min).
If pip asks you to **restart the session** (numpy version), do it, then re-run the first cell and continue from section 2.

In [ ]:
!apt-get -qq update && apt-get -qq install -y colmap ffmpeg > /dev/null
!pip -q install nerfstudio
!colmap -h | head -n 2; ns-train --help | head -n 3

## 2. Frames: 5 per second from every clip, then drop the blurriest 15%
Blur score = variance of the Laplacian (low = blurry). Motion blur from walking around hurts both COLMAP and the splat.

In [ ]:
import cv2, numpy as np, shutil
FR = '/content/frames'; shutil.rmtree(FR, ignore_errors=True); os.makedirs(FR)
for k, c in enumerate(CLIPS):
    !ffmpeg -loglevel error -i "{c}" -vf fps=5 -q:v 2 {FR}/clip{k}_%04d.jpg
imgs = sorted(glob.glob(f'{FR}/*.jpg'))
score = {p: cv2.Laplacian(cv2.imread(p, 0), cv2.CV_64F).var() for p in imgs}
cut = np.percentile(list(score.values()), 15)
for p, s in score.items():
    if s < cut: os.remove(p)
print(len(imgs), 'frames extracted,', len(glob.glob(f'{FR}/*.jpg')), 'kept after blur filter')

## 3. Camera poses with COLMAP
Vocabulary-tree matching scales to several clips (exhaustive matching of ~300 images on CPU is too slow).
Check the printed number of registered images: it should be close to the number of frames. If many are missing, re-run with `--matching-method exhaustive` (slower but more robust).

In [ ]:
!ns-process-data images --data /content/frames --output-dir /content/bee --matching-method vocab_tree --no-gpu --num-downscales 2
import json
T = json.load(open('/content/bee/transforms.json'))
print('registered images:', len(T['frames']), 'of', len(glob.glob('/content/frames/*.jpg')))

## 4. Train splatfacto (3DGS)
Half resolution (540×960) is plenty for a 5 cm object and trains in ~10-15 min on an A100. Training metrics go to TensorBoard; no live viewer needed.

In [ ]:
!ns-train splatfacto --data /content/bee --max-num-iterations 20000    --pipeline.datamanager.camera-res-scale-factor 0.5    --viewer.quit-on-train-completion True --vis tensorboard --output-dir /content/outputs
CFG = sorted(glob.glob('/content/outputs/bee/splatfacto/*/config.yml'))[-1]; print(CFG)

## 5. Export the splat + render an orbit preview, copy everything to Drive
`splat.ply` = the Gaussians (positions, scales, rotations, opacity, SH colour). `transforms.json` = the COLMAP cameras, needed to align the splat to the simulation and to render it from the press camera later.

In [ ]:
!ns-export gaussian-splat --load-config {CFG} --output-dir /content/export
!ns-render interpolate --load-config {CFG} --output-path /content/export/orbit_preview.mp4 --frame-rate 30 --interpolation-steps 4 --downscale-factor 2
OUT = f'{DRIVE}/splat_out'; os.makedirs(OUT, exist_ok=True)
!cp /content/export/*.ply {OUT}/splat.ply
!cp /content/export/orbit_preview.mp4 /content/bee/transforms.json {OUT}/
!cp -r /content/bee/colmap/sparse/0 {OUT}/colmap_sparse 2>/dev/null || true
!ls -la {OUT}

Done. Download `MyDrive/dirac_bee/splat_out/` (splat.ply, transforms.json, orbit_preview.mp4) into the project folder as `splat/` and tell Claude.